In [2]:
import torch

from edge_detector.scripts.benchmark import (
    benchmark_module,
    configure_cpu,
)
from edge_detector.scripts.bench_blocks import (
    build_benchmark_block,
    build_benchmark_transition,
)


THREADS = 4
WARMUP = 1
RUNS = 3

configure_cpu(THREADS)

In [3]:
BLOCK_NAMES = {
    "shufflenet": "InvertedResidual",
    "fasternet": "MLPBlock + PartialConv",
    "repvgg": "RepVGGBlock",
    "c2f": "C2f",
    "uib": "UIB",
    "ghost": "GhostBottleneck",
}

BLOCK_CONFIGS = {
    "c3": {
        "channels": 64,
        "height": 80,
        "width": 80,
    },
    "c4": {
        "channels": 128,
        "height": 40,
        "width": 40,
    },
    "c5": {
        "channels": 256,
        "height": 20,
        "width": 20,
    },
}

TRANSITION_CONFIGS = {
    "c3_c4": {
        "in_channels": 64,
        "out_channels": 128,
        "height": 80,
        "width": 80,
    },
    "c4_c5": {
        "in_channels": 128,
        "out_channels": 256,
        "height": 40,
        "width": 40,
    },
}

In [4]:
def benchmark_block(
    block_name: str,
    config_name: str,
    config: dict,
) -> dict:
    block = build_benchmark_block(
        block_name=block_name,
        channels=config["channels"],
    ).eval()

    x = torch.randn(
        1,
        config["channels"],
        config["height"],
        config["width"],
        dtype=torch.float32,
    )

    with torch.inference_mode():
        y = block(x)

    assert tuple(y.shape) == tuple(x.shape), (
        f"{block_name} / {config_name}: "
        f"expected {tuple(x.shape)}, got {tuple(y.shape)}"
    )

    result = benchmark_module(
        block,
        x,
        warmup=WARMUP,
        runs=RUNS,
    )

    return {
        "block": block_name,
        "name": BLOCK_NAMES[block_name],
        "config": config_name,
        **result.as_dict(),
    }


def benchmark_transition(
    block_name: str,
    config_name: str,
    config: dict,
) -> dict:
    block = build_benchmark_transition(
        block_name=block_name,
        in_channels=config["in_channels"],
        out_channels=config["out_channels"],
    ).eval()

    x = torch.randn(
        1,
        config["in_channels"],
        config["height"],
        config["width"],
        dtype=torch.float32,
    )

    expected_shape = (
        1,
        config["out_channels"],
        config["height"] // 2,
        config["width"] // 2,
    )

    with torch.inference_mode():
        y = block(x)

    assert tuple(y.shape) == expected_shape, (
        f"{block_name} / {config_name}: "
        f"expected {expected_shape}, got {tuple(y.shape)}"
    )

    result = benchmark_module(
        block,
        x,
        warmup=WARMUP,
        runs=RUNS,
    )

    return {
        "block": block_name,
        "name": BLOCK_NAMES[block_name],
        "config": config_name,
        **result.as_dict(),
    }

In [5]:
results = []

for block_name in BLOCK_NAMES:
    for config_name, config in BLOCK_CONFIGS.items():
        results.append(
            benchmark_block(
                block_name=block_name,
                config_name=config_name,
                config=config,
            )
        )

    for config_name, config in TRANSITION_CONFIGS.items():
        results.append(
            benchmark_transition(
                block_name=block_name,
                config_name=config_name,
                config=config,
            )
        )


CONFIG_ORDER = [
    "c3",
    "c4",
    "c5",
    "c3_c4",
    "c4_c5",
]

lookup = {
    (result["block"], result["config"]): result["median_ms"]
    for result in results
}

header = (
    f"{'Block':24s} "
    f"{'C3, ms':>10s} "
    f"{'C4, ms':>10s} "
    f"{'C5, ms':>10s} "
    f"{'C3->C4, ms':>12s} "
    f"{'C4->C5, ms':>12s}"
)

print(header)
print("-" * len(header))

for block_name in BLOCK_NAMES:
    values = [
        lookup[(block_name, config_name)]
        for config_name in CONFIG_ORDER
    ]

    print(
        f"{BLOCK_NAMES[block_name]:24s} "
        f"{values[0]:10.3f} "
        f"{values[1]:10.3f} "
        f"{values[2]:10.3f} "
        f"{values[3]:12.3f} "
        f"{values[4]:12.3f}"
    )

Block                        C3, ms     C4, ms     C5, ms   C3->C4, ms   C4->C5, ms
-----------------------------------------------------------------------------------
InvertedResidual              1.994      2.454      1.611        2.576        6.279
MLPBlock + PartialConv        6.141      3.080      2.615        3.419        3.070
RepVGGBlock                   2.738      2.769      2.886        6.454        5.742
C2f                           5.497      4.157      3.821       10.538        5.560
UIB                           5.436      3.185      2.510        7.521        8.537
GhostBottleneck               6.999      8.776      7.150       20.027       18.376
